In [ ]:
import pandas as pd
from tqdm import tqdm
import emoji

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

/usr/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.random.manual_seed(333)
root_path = "/home/stefan/ioai-prep/kits/pre-iaio/hieroglyph"

# Data

In [3]:
values = pd.read_csv(f"{root_path}/train_data.csv")["text"]
queries = pd.read_csv(f"{root_path}/test_data.csv")["emoji_sequence"]

# Model

In [4]:
model_path = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side='left', trust_remote_code=True)
model = AutoModel.from_pretrained(model_path, trust_remote_code=True)

model.to(device)
model.eval()

Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [5]:
# from https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
# much better than mean pooling for this model! 45/100 otherwise
def last_token_pool(last_hidden_states, attention_mask):
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[
            torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths
        ]

In [6]:
def compute_embedding(texts: list, is_query: bool = False):
    if isinstance(texts, str):
        texts = [texts]

    if is_query:
        task_description = "Given a sequence of emojis, retrieve the English sentence that best explains their meaning"
        texts = [f"Instruct:\n{task_description}\nQuery:\n{t}" for t in texts]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = last_token_pool(
            outputs.last_hidden_state, inputs["attention_mask"]
        )
        embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [7]:
old_clock = compute_embedding("🕰️")
new_clock = compute_embedding("🕛")
leg = compute_embedding("🦵")

torch.cosine_similarity(old_clock, new_clock), torch.cosine_similarity(old_clock, leg)

(tensor([0.7586], device='cuda:0'), tensor([0.5120], device='cuda:0'))

In [8]:
values_embd = []

for value_idx in tqdm(range(len(values))):
    value = str(values.iloc[value_idx]).strip()
    embd = compute_embedding(value)
    values_embd.append(embd)

  0%|          | 0/688 [00:00<?, ?it/s]

100%|██████████| 688/688 [00:20<00:00, 33.85it/s]


In [9]:
values_embd = torch.cat(values_embd)
values_embd.shape

torch.Size([688, 1024])

In [ ]:
sentence_ids = pd.read_csv(f"{root_path}/train_data.csv")["sentence_id"].tolist()
answers = []

for query_idx in tqdm(range(len(queries))):
    query = queries.iloc[query_idx]
    # 0.75 acc with 90/100 if we only feed the raw query
    query = ' '.join([emoji.demojize(e) for e in list(query)])
    
    query = compute_embedding([query], is_query=True)

    sim = query @ values_embd.T
    idx = torch.topk(sim, k=1).indices.cpu().item()
    answers.append(sentence_ids[idx])

100%|██████████| 64/64 [00:01<00:00, 32.48it/s]


# Submission

In [22]:
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": [f"query_{i:03d}" for i in range(1, 65)],
    "answer": answers
})


submission.head()

,subtaskID,datapointID,answer
0,1,query_001,1232
1,1,query_002,1527
2,1,query_003,1192
3,1,query_004,1240
4,1,query_005,1382


In [23]:
submission.to_csv(f"{root_path}/submission.csv", index=False)